# LangChain (open source): Q&A over Documents

### Outline
- Load a small product catalog as documents
- Build a vector store and retriever over it
- Manual "stuff retrieved docs into the prompt" answer
- The `RetrievalQA` equivalent: an LCEL RAG chain

Open-source recode of `02-LangChain-for-LLM-Application-Development/L4-QnA.ipynb`.
`VectorstoreIndexCreator`, `RetrievalQA`, `CSVLoader` and `DocArrayInMemorySearch` all live in
`langchain_community`/`langchain.chains`, neither of which is installed (or, for the chains,
even exists) in this LangChain version anymore:

- `CSVLoader` -> `common.load_csv_documents` (stdlib `csv`, no `langchain_community` needed).
- `DocArrayInMemorySearch` -> `langchain_core.vectorstores.InMemoryVectorStore`.
- `RetrievalQA` -> a small LCEL RAG chain: format whatever the retriever returns straight into
  the prompt's `{context}`.

Real embeddings would normally come from `OllamaEmbeddings`, but the Ollama cloud account behind
this project's `.env` returns "unauthorized" for every embedding model tried.
`local_embeddings.LocalTfidfEmbeddings` is a real (if simple) local TF-IDF vectorizer plugged
into the same `Embeddings` interface instead - see that module's docstring for the full story.

`OutdoorClothingCatalog_1000.csv` from the original isn't included in this repo;
`data/outdoor_clothing_catalog.csv` is a small synthetic stand-in with the same shape (a handful
of items explicitly UPF/sun-protection rated, the rest not, so retrieval is meaningful).

## Setup

This notebook is the open-source / Ollama-cloud recode of the matching lesson in
[`openai_agentic_ai_course`](../../openai_agentic_ai_course/), reusing the shared
`common.py` / `tracing.py` helpers already built for
[`open_source_agentic_ai_course`](../../open_source_agentic_ai_course/) rather than duplicating them here.

- **Model**: `ChatOllama`, pointed at the Ollama cloud endpoint configured in the
  repo-root `.env` (`OLLAMA_MODEL` / `OLLAMA_BASE_URL` / `OLLAMA_API_KEY`).
- **Tracing**: every `.invoke()` / `.batch()` / `.stream()` call below passes
  `config=traced("run name")`, which attaches a Langfuse callback - open the
  Langfuse dashboard and filter by trace name to see this notebook's calls.
- **Kernel**: run this with the repo's `.venv` (`Python 3 (ipykernel)`) - it already
  has everything in [`requirements.txt`](../../requirements.txt) installed.

In [1]:
import sys
from pathlib import Path

# common.py / tracing.py live in open_source_agentic_ai_course/, not here - add it to sys.path
# instead of copying them, so this notebook always uses the one shared implementation.
COURSE_DIR = Path("../../open_source_agentic_ai_course").resolve()
if str(COURSE_DIR) not in sys.path:
    sys.path.insert(0, str(COURSE_DIR))

from common import get_model, load_csv_documents, traced
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.vectorstores import InMemoryVectorStore
from local_embeddings import LocalTfidfEmbeddings

In [2]:
docs = load_csv_documents(str(COURSE_DIR / "data" / "outdoor_clothing_catalog.csv"))
print(f"loaded {len(docs)} catalog rows")
docs[0]

loaded 15 catalog rows


Document(metadata={'source': 'C:\\Users\\Stefan.Duprey\\Documents\\GitHub\\langchain_courses\\open_source_agentic_ai_course\\data\\outdoor_clothing_catalog.csv', 'row': 0}, page_content='name: Sun Shield Shirt\ndescription: High-performance sun shirt with UPF 50+ sun protection, blocking 98% of harmful UV rays. Slightly fitted, falls at hip. Moisture-wicking, abrasion-resistant fabric. Machine washable and dryable. Recommended by The Skin Cancer Foundation.')

In [3]:
model = get_model()

vectorstore = InMemoryVectorStore.from_documents(docs, LocalTfidfEmbeddings())

A quick sanity check: does similarity search actually surface the sun-protection items for a
sunblocking query?

In [4]:
query = "Please suggest a shirt with sunblocking"
hits = vectorstore.similarity_search(query, k=4)
print(f"{len(hits)} hits for {query!r}")
for doc in hits:
    print(" -", doc.page_content.splitlines()[0])

4 hits for 'Please suggest a shirt with sunblocking'
 - name: Sun Shield Shirt
 - name: Men's Tropical Plaid Short-Sleeve Shirt
 - name: Refresh Swimwear V-Neck Tankini
 - name: Men's Classic Denim


In [5]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})


def format_docs(retrieved) -> str:
    return "\n\n".join(doc.page_content for doc in retrieved)

## The manual version

Retrieve, then stuff every retrieved doc straight into one prompt - what `RetrievalQA` did under
the hood before it was removed.

In [6]:
full_query = "Please list all your shirts with sun protection in a table in markdown and summarize each one."
qdocs = format_docs(retriever.invoke(full_query))
manual_response = model.invoke(
    f"{qdocs}\n\nQuestion: {full_query}", config=traced("L4: manual stuffed-context answer")
)
print(manual_response.content)

| Product | Summary |
|---------|---------|
| **Women's Sun Shade Hoodie** | Featherweight hoodie with UPF 50+ protection, thumbholes for hand coverage, and a relaxed fit ideal for paddling, hiking, or beach days. |
| **Refresh Swimwear V‑Neck Tankini** | Chlorine‑resistant tankini top featuring UPF 50+ built‑in sun protection, racerback straps for a secure fit, and designed to pair with matching swim bottoms. |

These are the only shirts in the catalog that include sun‑protective features.


## The `RetrievalQA` equivalent: an LCEL RAG chain

Same idea, expressed as a reusable chain instead of a one-off string. `RunnablePassthrough.assign`
adds a `context` key built from the retriever's output, then the prompt/model/parser run as
usual.

In [7]:
rag_prompt = ChatPromptTemplate.from_template(
    "Use the following product catalog excerpts to answer the question. "
    "If the answer isn't in the excerpts, say you don't know.\n\n"
    "{context}\n\nQuestion: {question}"
)

qa_chain = (
    RunnablePassthrough.assign(context=lambda x: format_docs(retriever.invoke(x["question"])))
    | rag_prompt
    | model
    | StrOutputParser()
)

print(qa_chain.invoke({"question": full_query}, config=traced("L4: RAG chain answer")))

| Shirt | Sun Protection | Summary |
|-------|-----------------|---------|
| **Women's Sun Shade Hoodie** | UPF 50+ | Featherweight hoodie with built‑in UPF 50+ protection and thumbholes to cover the backs of your hands. Ideal for paddling, hiking, or a day at the beach. |
| **Refresh Swimwear V‑Neck Tankini** | UPF 50+ | Chlorine‑resistant tankini top with built‑in UPF 50+ sun protection. Racerback straps provide a secure fit and it pairs with our swim bottoms. |

*Only the items above are shirts that explicitly mention sun protection in the catalog excerpts.*
